In [1]:
from pathlib import Path
from typing import Optional
import os
from concurrent.futures import ThreadPoolExecutor
import pandas as pd, numpy as np
from utils import *

In [2]:
df_simule = pd.DataFrame(pd.read_csv(r"../../datas/bronze/dataset_brut.csv"))
df_maintenance = pd.DataFrame(pd.read_csv(r"../../datas/gold/postgres_maintenance.csv"))
df_alerte = pd.DataFrame(pd.read_csv(r"../../datas/gold/postgres_alerte.csv"))

In [3]:
df_alert_complet, df_maintenance_complet  = attach_alerte_maintenance_ids(
    df_simule=df_simule,
    df_maintenance=df_maintenance,
    df_alerte=df_alerte
)

In [4]:
df_alert_complet.head()
df_alert_filtered = df_alert_complet[["timestamp", "machine_id", "id_alerte"]]

In [5]:
df_maintenance_complet.head()
df_maintenance_filtered = df_maintenance_complet[["timestamp", "machine_id", "id_maintenance"]]

In [6]:
df_alert_filtered = sort_dataframe_by_timestamp(df_alert_filtered)
df_maintenance_filtered = sort_dataframe_by_timestamp(df_maintenance_filtered)

In [7]:
df_maintenance_clean = build_episode_dataframe(
    df_maintenance_filtered,
    id_column="id_maintenance",
    id_output_column="id_panne",
    start_column="debut_panne",
    end_column="fin_panne",
)
df_alert_clean = build_episode_dataframe(
    df_alert_filtered,
    id_column="id_alerte",
    id_output_column="id_alerte",
    start_column="debut_alerte",
    end_column="fin_alerte",
)

In [8]:
folder_path = Path("../../datas/gold")
df_maintenance_clean.to_csv(
    name_csv_file(
        folder_path=folder_path,
        filename="maintenance",
        extension=".csv",
        type_dst="influxdb"
    ), index=False, encoding='utf-8'
)
df_alert_clean.to_csv(
    name_csv_file(
        folder_path=folder_path,
        filename="alerte",
        extension=".csv",
        type_dst="influxdb"
    ), index=False, encoding='utf-8'
)